# Task 4 - Step 4: Export results

Copies tables to `results/task4/` and figures to `figures/task4/`, and builds `results/task4/results.json`
(raw values, per-epoch histories, configs, freeze manifest, unknown-access log).

In [1]:
# ---- Task 4 common header (identical in every Task 4 notebook) ----
# NOTE: no CIFAR-100 image is decoded or loaded anywhere except notebook 03, after the freeze check.
import json, os, sys, time, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

REPO = Path.cwd().resolve()
while not (REPO / "task4" / "training.py").exists():
    if REPO.parent == REPO:
        raise RuntimeError("Run this notebook from inside the PA1 repository")
    REPO = REPO.parent
sys.path.insert(0, str(REPO))

from shared.config import load_config
from task4.data import cifar10
from task4.models.resnet_cifar import build_model

T4 = REPO / "task4"
CFG_DIR = T4 / "configs"
SEED = 6304
# Smoke mode (env TASK4_SMOKE=1): 2 epochs on 2000 training images; outputs under _smoke/; notebook 03 uses
# RANDOM stand-in "unknown" images, so no CIFAR-100 image is touched.
SMOKE = os.environ.get("TASK4_SMOKE", "0") == "1"
SUB = "_smoke" if SMOKE else ""
RES = T4 / "results" / SUB
TAB, FIG = RES / "tables", RES / "figures"
DATA_TAB = T4 / "results" / "tables"       # data-preparation files (same path in smoke and real mode)
DATA_TAB.mkdir(parents=True, exist_ok=True)
CKPT = T4 / "checkpoints" / SUB            # git-ignored
CACHE = T4 / "cache" / SUB                 # git-ignored
for p in [TAB, FIG, CKPT, CACHE]:
    p.mkdir(parents=True, exist_ok=True)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

RUNS = ["vanilla", "gcsc", "proser"]
LABEL = {"vanilla": "Vanilla", "gcsc": "GCSC", "proser": "PROSER"}
SCORES = ["msp", "mls", "energy", "mahalanobis"]         # post-hoc scores on the frozen Vanilla model
SCORE_LABEL = {"msp": "MSP", "mls": "MLS", "energy": "Energy", "mahalanobis": "Mahalanobis",
               "placeholder": "PROSER placeholder"}


def run_dir(n):
    return CKPT / load_config(CFG_DIR, n)["run_name"]


def load_trained(n):
    """Frozen model with the selected checkpoint of run ``n``."""
    cfg = load_config(CFG_DIR, n)
    ck = torch.load(run_dir(n) / "best.pt", map_location="cpu", weights_only=False)
    model = build_model(cfg["model"]["num_classes"], cfg["model"]["dummy_classifiers"], cfg["seed"])
    model.load_state_dict(ck["model"])
    return model.to(DEVICE).eval(), ck, cfg


@torch.no_grad()
def extract(model, x_u8, batch_size=512):
    """Penultimate features and logits of a frozen model (no augmentation)."""
    feats, logits = [], []
    for s in range(0, len(x_u8), batch_size):
        f, z = model(cifar10.normalize(x_u8[s:s + batch_size].to(DEVICE)))
        feats.append(f.float().cpu()); logits.append(z.float().cpu())
    return torch.cat(feats), torch.cat(logits)


plt.rcParams.update({"font.size": 11, "axes.titlesize": 12, "legend.fontsize": 9, "savefig.dpi": 150})


def savefig(fig, stem):
    fig.savefig(FIG / f"{stem}.png", bbox_inches="tight")
    fig.savefig(FIG / f"{stem}.pdf", bbox_inches="tight")
    print("saved figure", stem)


print("REPO:", REPO, "| device:", DEVICE, "| smoke:", SMOKE)

REPO: C:\Users\afifh\Desktop\ATML\PA1 | device: cuda | smoke: False


In [2]:
import math, shutil
R_OUT = (RES / "export") if SMOKE else (REPO / "results" / "task4")
F_OUT = (RES / "export" / "figures") if SMOKE else (REPO / "figures" / "task4")
R_OUT.mkdir(parents=True, exist_ok=True); F_OUT.mkdir(parents=True, exist_ok=True)
copied = []
for p in sorted(set(TAB.glob("task4_*")) | set(DATA_TAB.glob("task4_*"))):
    shutil.copy2(p, R_OUT / p.name); copied.append(p.name)
for p in sorted(FIG.glob("task4_*")):
    shutil.copy2(p, F_OUT / p.name); copied.append("figures/" + p.name)
print(len(copied), "files copied")

18 files copied


In [3]:
def clean(o):
    if isinstance(o, float) and (math.isnan(o) or math.isinf(o)):
        return None
    if isinstance(o, dict):
        return {str(k): clean(v) for k, v in o.items()}
    if isinstance(o, list):
        return [clean(v) for v in o]
    if isinstance(o, (np.floating, np.integer, np.bool_)):
        return o.item()
    return o

def csv(name):
    return pd.read_csv(TAB / name)

log_path = REPO / "data" / "cifar" / "unknown_access_log.jsonl"
res = {"task": "task4_open_set_recognition", "seed": SEED,
       "data": json.loads((DATA_TAB / "task4_data_summary.json").read_text()),
       "closed_set_accuracy": csv("task4_closed_set_accuracy.csv").to_dict("records"),
       "thresholds": csv("task4_thresholds.csv").to_dict("records"),
       "score_comparison_vanilla": csv("task4_score_comparison.csv").to_dict("records"),
       "model_comparison_mls": csv("task4_osr_metrics.csv").to_dict("records"),
       "unknown_class_analysis": csv("task4_unknown_class_analysis.csv").to_dict("records"),
       "failure_cases": csv("task4_failure_cases.csv").to_dict("records"),
       "freeze_manifest": json.loads((CACHE / "freeze_manifest.json").read_text()),
       "unknown_access_log": [json.loads(l) for l in log_path.read_text().splitlines()] if log_path.exists() else None,
       "runs": {}, "_notes": {"unknownness": "every score is an unknownness u(x); accept when u(x) <= tau",
                              "threshold": "tau = 95th percentile of u on CIFAR-10 validation only",
                              "proser_csa": "computed from the ten known-class logits only",
                              "rpl": "optional extension; not implemented"}}
for n in RUNS:
    d = run_dir(n)
    res["runs"][n] = {"label": LABEL[n], "config": load_config(CFG_DIR, n),
                      "training_summary": json.loads((d / "summary.json").read_text()),
                      "history": json.loads((d / "history.json").read_text())}
res["files"] = sorted(copied)
(R_OUT / "results.json").write_text(json.dumps(clean(res), indent=1))
print("wrote", R_OUT / "results.json")
print(csv("task4_osr_metrics.csv")[["model", "score", "cifar10_test_csa", "auroc_known_vs_near", "auroc_known_vs_far",
                                    "near_rejection_rate", "far_rejection_rate"]].round(4).to_string(index=False))

wrote C:\Users\afifh\Desktop\ATML\PA1\results\task4\results.json
  model              score  cifar10_test_csa  auroc_known_vs_near  auroc_known_vs_far  near_rejection_rate  far_rejection_rate
Vanilla                MLS            0.9461               0.7921              0.8983               0.3088              0.5162
   GCSC                MLS            0.9543               0.8337              0.9100               0.3750              0.6088
 PROSER                MLS            0.9440               0.8006              0.8637               0.3025              0.4225
 PROSER PROSER placeholder            0.9440               0.7868              0.8874               0.2900              0.4600
